# CZ Gate Calibration with Tunable Couplers

## Prerequisites
This guide assumes you have a configured `DeviceSetup` as well as `Qubit` objects with assigned parameters. Before you can calibrate a CZ gate, you need to have tuned up the single-qubit drive and readout of both qubits, and to have parked the tunable coupler at a bias that minimizes the residual ZZ coupling (see the [ZZ coupling guide](15_zz_coupling.ipynb)). Please see [our tutorials](https://docs.zhinst.com/labone_q_user_manual/applications_library/tutorials/index.html) if you need to create your setup and qubits for the first time.

You can run this notebook on real hardware in the lab. However, if you don't have the hardware at your disposal, you can also run the notebook "as is" using an emulated session (see below).

If you are just getting started with the LabOne Q Applications Library, please don't hesitate to reach out to us at info@zhinst.com.

## Background

In this guide, you will learn how to calibrate a CZ gate between a pair of `TunableTransmonQubits` mediated by a `TunableCoupler`.

### The gate mechanism

The CZ gate calibrated here is a dynamical flux gate: it realizes a conditional phase of $\pi$ on the $|11\rangle$ state by tuning the $|11\rangle$ state into resonance with the $|20\rangle$ state for a well-defined duration. During this time, the $|11\rangle$ state makes a full round-trip in the two-state manifold (Bloch sphere) spanned by $|11\rangle$ and $|20\rangle$, thereby acquiring a geometric phase of $\pi$ [1, 2]. Because the round-trip must be complete — any residual population left in $|20\rangle$ at the end of the pulse is leakage out of the computational subspace — both the amplitude (which sets the detuning from the $|11\rangle \leftrightarrow |20\rangle$ resonance) and the duration of the flux pulse have to be calibrated precisely.

In the implementation used here, the interaction is activated parametrically [3]: a flux pulse modulated at the parametric-drive frequency (the `modulated_flux` pulse) is played on the flux line of the qubit with the higher ge frequency, while a simultaneous baseband pulse on the coupler flux line brings the coupling strength up for the duration of the gate. The gate ends with single-qubit phase corrections (virtual-Z rotations) on both qubits.

The `cz` quantum operation of `TunableCouplerOperations` implements this sequence. The parameters of the gate are stored on a `QPU` topology edge tagged `"cz"`, whose parameters are an instance of `CzParameters`:

* `control_pulse`: the fast-flux pulse played on the qubit flux line,
* `coupler_pulse`: the fast-flux pulse played on the coupler flux line,
* `control_angle` and `target_angle`: the single-qubit phase corrections.

Note that the same `cz` operation also supports a *coupler-driven* flavor, in which the modulated flux pulse is played on the coupler itself and no qubit flux pulse is used. That flavor is selected by leaving `control_pulse` set to `None` (the default), and its calibration is covered by the `para_cz_*` workflows in `laboneq_applications.contrib.experiments.two_qubit`. This guide covers the *qubit-driven* flavor, where `control_pulse` is set.

### The calibration procedure

The calibration proceeds in three steps, each one an experiment `Workflow` from `laboneq_applications.contrib.experiments`:

1. **Chevron** (`chevron`): sweep the duration and the amplitude of the fast-flux pulses while both qubits are prepared in $|1\rangle$. The characteristic chevron pattern of the $|11\rangle \leftrightarrow |20\rangle$ avoided crossing reveals the operating point $(a_{CZ}, l_{CZ})$ of the gate.
2. **JAZZ** (`jazz`): a Ramsey-type sequence with two CZ gates and an echo pulse, used to fine-tune the conditional-phase accumulation of the gate to the target value of $\pi$.
3. **Phase correction** (`phase_correction`): Ramsey-type measurements on the control and target qubits that determine the single-qubit phase corrections `control_angle` and `target_angle`.

## Imports

You'll start by importing `laboneq.simple`.

In [ ]:
from laboneq.simple import *

## Define your experimental setup

Let's define our experimental setup. We will need:

* a [DeviceSetup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html)

* 2 [TunableTransmonQubits](https://docs.zhinst.com/labone_q_user_manual/applications_library/reference/qpu_types/tunable_transmon.html#laboneq_applications.qpu_types.tunable_transmon.TunableTransmonQubit)

* a `TunableCoupler` connecting the two qubits

* a set of `TunableCouplerOperations`

* a [QPU](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/04_quantum_processing_unit/concepts/index.html) with a topology containing `"cz"` edges

Here, we will show explicitly how to build these objects, since the `"cz"` topology edges and their `CzParameters` are the essential new ingredient compared to the single-qubit guides.

In [ ]:
number_of_qubits = 2

### DeviceSetup

This guide requires a setup that can drive and readout two tunable transmon qubits, and that provides flux lines for the qubits and the tunable coupler. Here, we will use an SHFQC+ with 6 signal generation channels, an HDAWG with 8 channels, and a PQSC.

If you have used LabOne Q before and already have a `DeviceSetup` for your setup, you can reuse that.

If you do not have a `DeviceSetup`, you can create one using the code below. Just change the device numbers to the ones in your rack and adjust any other input parameters as needed.

In [ ]:
from laboneq.contrib.example_helpers.generate_device_setup import generate_device_setup

setup = generate_device_setup(
    number_qubits=number_of_qubits,
    pqsc=[{"serial": "DEV10001"}],
    shfqc=[
        {
            "serial": "DEV12001",
            "number_of_channels": 6,
            "readout_multiplex": 6,
            "options": "SHFQC/PLUS/QC6CH/RTR",
        }
    ],
    hdawg=[
        {"serial": "DEV8800", "number_of_channels": 8, "options": "HDAWG8/CNT/ME/PC"}
    ],
    include_flux_lines=True,
    server_host="localhost",
)

### Qubits

We will generate 2 `TunableTransmonQubits` from the logical signal groups in our `DeviceSetup`, and configure their parameters to reflect the properties of the qubits on your QPU.

In [ ]:
from laboneq_applications.qpu_types.tunable_transmon import TunableTransmonQubit

qubits = TunableTransmonQubit.from_device_setup(setup)

for q in qubits:
    q.parameters.ge_drive_pulse["sigma"] = 0.25
    q.parameters.readout_amplitude = 0.5
    q.parameters.reset_delay_length = 1e-6
    q.parameters.readout_range_out = -25
    q.parameters.readout_lo_frequency = 7.4e9

qubits[0].parameters.drive_lo_frequency = 6.4e9
qubits[0].parameters.resonance_frequency_ge = 6.3e9
qubits[0].parameters.resonance_frequency_ef = 6.0e9
qubits[0].parameters.readout_resonator_frequency = 7.0e9

qubits[1].parameters.drive_lo_frequency = 6.4e9
qubits[1].parameters.resonance_frequency_ge = 6.5e9
qubits[1].parameters.resonance_frequency_ef = 6.3e9
qubits[1].parameters.readout_resonator_frequency = 7.3e9

### Tunable Coupler

Let's now add the tunable coupler to our setup. We name the coupler after the qubits it connects and manually add its flux line to the setup. The connectivity will be formalized later in the `QPUTopology`.

In [ ]:
from laboneq_applications.qpu_types.tunable_coupler import TunableCoupler

# define desired couplings
couplings = {"c_q0q1": ("q0", "q1")}

# add signal lines for tunable couplers
for n, key in enumerate(couplings):
    # the first HDAWG channels are already occupied by the qubit flux lines
    channel_id = number_of_qubits + n
    if key in setup.logical_signal_groups:
        print(f"INFO: Logical signal for group {key} is already in setup")
    else:
        setup.add_connections(
            "hdawg_0",
            create_connection(to_signal=f"{key}/flux", ports=f"SIGOUTS/{channel_id}"),
        )

couplers = TunableCoupler.from_device_setup(setup, qubit_uids=couplings.keys())

### Quantum Operations

Create the set of `TunableCouplerOperations`. It extends `TunableTransmonOperations` with the coupler-mediated two-qubit operations we will calibrate in this guide: `cz`, as well as `iswap` and `discharge_coupler`. These operations look up the coupler and the gate parameters from the QPU topology edge connecting the two qubits, so no further operation needs to be defined.

In [ ]:
from laboneq_applications.qpu_types.tunable_coupler import TunableCouplerOperations

qops = TunableCouplerOperations()

### QPU

Now that we have all the ingredients, let's create the `QPU` object from the qubits, couplers and the quantum operations.

In [ ]:
from laboneq.dsl.quantum import QPU

qpu = QPU(qubits + couplers, quantum_operations=qops)

### QPU Topology

We now define the topology of this chip following the couplings defined before. For each coupler we add, in both directions:

* an edge tagged `"coupler"`, describing the physical connectivity;
* an edge tagged `"cz"` carrying an instance of `CzParameters`. This is where the `cz` operation and the calibration workflows below read and store the gate parameters.

We add the edges in both directions to support the fact that the coupler can act in both directions; each direction carries its own `CzParameters` instance, since for instance the phase corrections depend on which qubit plays the role of the control.

For the qubit-driven gate flavor calibrated in this guide, we explicitly set the `control_pulse` (the modulated fast-flux pulse on the qubit) and use a constant baseband pulse on the coupler:

In [ ]:
from laboneq_applications.qpu_types.tunable_coupler import CzParameters

for coupler, (q0, q1) in couplings.items():
    for source, target in ((q0, q1), (q1, q0)):
        qpu.topology.add_edge(
            source_node=source,
            target_node=target,
            quantum_element=coupler,
            tag="coupler",
        )
        qpu.topology.add_edge(
            source_node=source,
            target_node=target,
            quantum_element=coupler,
            tag="cz",
            parameters=CzParameters(
                control_pulse={
                    "function": "modulated_flux",
                    "amplitude": 0.0,
                    "length": 100e-9,
                    "frequency": 300e6,
                },
                coupler_pulse={
                    "function": "const",
                    "amplitude": 0.0,
                    "length": 100e-9,
                },
            ),
        )

Let's take a look at it by plotting the topology!

In [ ]:
qpu.topology.plot();

The `"cz"` edge now holds the qubit-driven gate parameters: the `modulated_flux` pulse (a Gaussian-square envelope modulated at the parametric-drive frequency) on the qubit flux line and a constant pulse on the coupler flux line:

In [ ]:
qpu.topology["cz", "q0", "q1"].parameters

### Alternatively, use the demo platform helper

The construction above is also available as a single helper function in the contrib library, which is convenient for testing and for trying out the workflows in emulation:

```python
from laboneq_applications.contrib.qpu_types import demo_platform_coupled_transmons

platform = demo_platform_coupled_transmons(n_qubits=2)
setup = platform.setup
qpu = platform.qpu
```

If you already have a `DeviceSetup` and a `QPU` stored in `.json` files, you can simply load them back using:

```python
from laboneq import serializers

setup = serializers.load(full_path_to_device_setup_file)
qpu = serializers.load(full_path_to_qpu_file)
```

## Connect to Session

In [ ]:
session = Session(setup)
session.connect(do_emulation=True)  # do_emulation=False when at a real setup

## Create a `FolderStore` for Saving Data

The experiment `Workflows` can automatically save the inputs and outputs of all their tasks to the folder path we specify when instantiating the `FolderStore`. Here, we choose the current working directory.

In [ ]:
# import FolderStore from the `workflow` namespace of LabOne Q, which was imported
# from `laboneq.simple`
from pathlib import Path

folder_store = workflow.logbook.FolderStore(Path.cwd())

We disable saving in this guide. To enable it, simply run `folder_store.activate()`.

In [ ]:
folder_store.deactivate()

## Optional: Configure the LoggingStore

You can also activate/deactivate the `LoggingStore`, which is used for displaying the `Workflow` logging information in the notebook; see again the [tutorial on Recording Experiment Workflow Results](../../../../tutorials/sources/logbooks.ipynb) for details.

Displaying the `Workflow` logging information is activated by default, but here we deactivate it to shorten the outputs, which are not very meaningful in emulation mode.

**We recommend that you do not deactivate the Workflow logging in practice.**

In [ ]:
from laboneq.workflow.logbook import LoggingStore

logging_store = LoggingStore()
logging_store.deactivate()

## Step 1: Chevron

You'll now run the chevron experiment workflow to find the operating point of the CZ gate. The experiment prepares both qubits in the $|1\rangle$ state using $\pi$-pulses, applies the pair of fast-flux pulses (modulated pulse on the qubit, baseband pulse on the coupler), and measures both qubits at the end of the sequence:

![Pulse sequence of the chevron experiment](../../../images/cz_chevron_sequence.svg "Pulse sequence of the chevron experiment.")

The workflow sweeps the duration of the fast-flux pulses (outer sweep) and the amplitude of one of the two pulses (inner sweep). The amplitude range should be chosen to bring the $|11\rangle$ and $|20\rangle$ states into resonance somewhere within the sweep, and the maximum duration should be large enough to observe at least one full population swap in the $|11\rangle$–$|20\rangle$ subspace. The pulse whose amplitude is swept is the one that drives the gate, so it follows the flavor of the edge: the *qubit* pulse (`control_pulse`) for the qubit-driven flavor used here, and the *coupler* pulse (`coupler_pulse`) on an edge that leaves `control_pulse` unset.

The measured qubit population forms the characteristic chevron pattern sketched below (simulation). From sinusoidal fits to horizontal slices of the data, one can determine the pulse amplitude $a_{CZ}$ with the smallest oscillation frequency — the point where $|11\rangle$ and $|20\rangle$ are exactly on resonance — and the duration $l_{CZ}$ at which the population has made its first full round-trip back into $|11\rangle$ for $a = a_{CZ}$. This point $(a_{CZ}, l_{CZ})$, marked with a diamond below, is the operating point of the gate:

![Simulated chevron pattern](../../../images/cz_chevron_pattern.svg "Simulated chevron pattern of the population left in the |11> state. The gate operating point (a_CZ, l_CZ) sits on the first full-revival fringe at the waist of the chevron.")

The current automated analysis uses a simpler and more robust criterion suitable as a starting point: it extracts the sweep point of maximum interaction (maximum population transfer on the target qubit). With `update(True)`, this `(duration, amplitude)` point is written into the `"cz"` edge, providing the starting point for the finer calibrations below. The duration goes to `coupler_pulse`, since that is where the `cz` operation reads the gate length from, and the amplitude to the pulse that drives the gate.

In [ ]:
import numpy as np

from laboneq_applications.contrib.experiments.two_qubit import chevron

options = chevron.experiment_workflow.options()
options.count(1024)
options.do_analysis(True)
options.update(True)

qubit_pairs = [["q0", "q1"]]

chevron_result = chevron.experiment_workflow(
    session=session,
    qpu=qpu,
    qubit_pairs=qubit_pairs,
    durations=[np.linspace(0.1e-6, 1.0e-6, 11) for _ in qubit_pairs],
    amplitudes=[np.linspace(0.1, 0.6, 11) for _ in qubit_pairs],
    options=options,
).run()

Let's inspect the pulse sequence of the experiment using `plot_simulation` and the output simulator:

In [ ]:
from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation

compiled_experiment = chevron_result.tasks["compile_experiment"].output
plot_simulation(compiled_experiment, start_time=0, length=10e-6)

The analysis workflow returns the old and new edge parameters. The entries are keyed by the edge signatures `"tag:source_node_uid:target_node_uid"`; the `update_qpu_edges` task converts these to the edge keys accepted by `qpu.update`. Since we ran with `update(True)`, the pulse parameters of the `"cz"` edge have already been updated:

In [ ]:
analysis_output = chevron_result.tasks["analysis_workflow"].output
print(analysis_output["new_parameter_values"])
edge_parameters = qpu.topology["cz", "q0", "q1"].parameters
print(edge_parameters.coupler_pulse)
print(edge_parameters.control_pulse)

Note that in emulation mode the measured data is random, so the extracted operating point is meaningless here — on real hardware you would see the chevron pattern in the population plots produced by the analysis.

## Step 2: JAZZ — conditional-phase calibration

Next, fine-tune the conditional phase of the gate. Conceptually, a conditional-phase measurement consists of two Ramsey-type experiments on one of the qubits, with the other qubit prepared once in $|0\rangle$ and once in $|1\rangle$. In each Ramsey experiment, the phase $\varphi$ of the second $\pi/2$-pulse is swept, and a sinusoidal fit to the population oscillation yields a phase $\delta_i$. The difference $\delta_1 - \delta_2$ between the two experiments is the conditional phase of the gate, with a target value of $\pi$.

The JAZZ sequence used here is an echo variant of this measurement: two CZ gates, each of half the duration, enclose an echo $\pi$-pulse on both qubits. The echo makes the sequence insensitive to the single-qubit dynamical phases and to low-frequency dephasing, so that the fitted oscillation directly reflects the conditional-phase accumulation:

![Pulse sequence of the JAZZ experiment](../../../images/cz_jazz_sequence.svg "Pulse sequence of the JAZZ experiment. Two half-duration CZ gates enclose an echo pulse on both qubits; the phase of the final pi/2-pulse is swept.")

The analysis fits the oscillation of the control-qubit population versus the flux-pulse amplitude and extracts the amplitude at which the gate accumulates a conditional phase of $\pi$. With `update(True)`, this amplitude is written into the pulse that drives the gate and the corresponding duration into the `coupler_pulse` of the edge. Here we keep the update disabled since the emulated data is random.

On real hardware, a robust procedure is to iterate [1]:

1. Measure the conditional phase for 10–20 values of the pulse amplitude $a$ centered around $a_{CZ}$, at fixed duration $l \approx l_{CZ}$, and set the amplitude that yields a conditional phase of $\pi$.
2. Measure the conditional phase and the leakage for 10–20 values of the pulse duration $l$ centered around $l_{CZ}$ (the conditional phase is expected to vary only little), and choose the duration that minimizes leakage. Leakage can be monitored by also reading out the qubit that passes through the $|2\rangle$ state during the gate.
3. Repeat step 1.

In [ ]:
from laboneq_applications.contrib.experiments.two_qubit import jazz

options = jazz.experiment_workflow.options()
options.count(1024)
options.do_analysis(True)

jazz_result = jazz.experiment_workflow(
    session=session,
    qpu=qpu,
    qubit_pairs=qubit_pairs,
    durations=[np.linspace(0.1e-6, 1.0e-6, 11) for _ in qubit_pairs],
    amplitudes=[np.linspace(0.1, 0.6, 11) for _ in qubit_pairs],
    options=options,
).run()

## Step 3: Single-qubit phase corrections

Finally, determine the single-qubit phase corrections. During the flux pulses, each qubit accumulates a dynamical phase (for instance from the ac-Stark shift of the parametric drive), which must be compensated so that the gate acts as a pure CZ in the computational frame. The compensation is applied as a virtual-Z rotation — precisely the `rz` operations the `cz` quantum operation plays at the end, with the angles taken from the `"cz"` edge parameters.

The measurement is again Ramsey-type: a $\pi/2$-pulse, the CZ gate, and a second $\pi/2$-pulse with swept phase, run once for the control and once for the target qubit (with the respective other qubit left in $|0\rangle$, so that no conditional phase contributes):

![Pulse sequence of the single-qubit phase-correction experiment](../../../images/cz_phase_correction_sequence.svg "Pulse sequence of the single-qubit phase-correction experiment, shown for one of the two qubits.")

The analysis fits a cosine to both oscillations and extracts the phase offsets. With `update(True)`, these are written into the `control_angle` and `target_angle` parameters of the `"cz"` edge, and the `cz` operation will apply them automatically from then on.

In [ ]:
from laboneq_applications.contrib.experiments.two_qubit import phase_correction

options = phase_correction.experiment_workflow.options()
options.count(1024)
options.do_analysis(True)

phase_result = phase_correction.experiment_workflow(
    session=session,
    qpu=qpu,
    qubit_pairs=qubit_pairs,
    phases=[np.linspace(0, 2 * np.pi, 21) for _ in qubit_pairs],
    options=options,
).run()

In [ ]:
phase_analysis = phase_result.tasks["analysis_workflow"].output
phase_analysis["new_parameter_values"]

## Wrapping up

The `"cz"` edge now holds the full set of gate parameters:

In [ ]:
qpu.topology["cz", "q0", "q1"].parameters

After these three steps on real hardware, the calibrated gate can be played simply as:

```python
qop = qpu.quantum_operations
qop.cz(qpu["q0"], qpu["q1"])
```

To assess the fidelity of the calibrated gate, continue with the two-qubit interleaved randomized benchmarking workflow, `laboneq_applications.contrib.experiments.two_qubit.two_qubit_irb`.

## References

[1] M. A. Rol et al., *Fast, High-Fidelity Conditional-Phase Gate Exploiting Leakage Interference in Weakly Anharmonic Superconducting Qubits*, Phys. Rev. Lett. 123, 120502 (2019), [arXiv:1903.02492](https://arxiv.org/abs/1903.02492).

[2] S. Krinner et al., *Realizing repeated quantum error correction in a distance-three surface code*, [Nature 605, 669 (2022)](https://www.nature.com/articles/s41586-022-04566-8).

[3] N. Lacroix et al., *Improving the Performance of Deep Quantum Optimization Algorithms with Continuous Gate Sets*, PRX Quantum 1, 020304 (2020), [arXiv:2005.05275](https://arxiv.org/abs/2005.05275).